# 01 · M5 dataset understanding and deterministic series selection

Extends the original structural inspection with missingness/key audits, overlap verification and reproducible selection. All **analytical target inspection and samples stop at d_1801**. The mandated validation/evaluation overlap audit compares only d_1–d_1913; final holdout targets d_1914–d_1941 are never previewed or analyzed. Shapes, column names and date boundaries describe file structure only.

Run after environment setup. This notebook writes the source dictionary/audit and recomputes `data/interim/selected_skus.csv`; it never edits raw CSVs. Raw type declarations are explicit in reusable loaders. Target families are compressed as inclusive consecutive column ranges for concise output.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

cwd = Path.cwd().resolve()
ROOT = next(p for p in (cwd, *cwd.parents) if (p / "experiment.yaml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import load_config, data_path
cfg = load_config()
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.max_rows", 25)
print(f"Python {sys.version.split()[0]}, pandas {pd.__version__}")


Python 3.14.6, pandas 3.0.6


## Load and validate sources

`audit_raw()` reads the existing CSVs, validates sales/calendar/price keys, reports missingness within stated scopes, and compares overlapping sales. It never concatenates the sales histories. The comparison excludes the suffix-dependent literal `id`.

In [2]:
from src.data.understanding import audit_raw
from src.data.select_skus import run_selection

audit = audit_raw(cfg)
selected, selection = run_selection(cfg)
display(pd.DataFrame([
    {"dataset": name, "rows": info["shape"][0], "columns": info["shape"][1]}
    for name, info in audit["files"].items()
]).set_index("dataset"))

,rows,columns
dataset,,
calendar.csv,1969,14
sell_prices.csv,6841121,4
sales_train_validation.csv,30490,1919
sales_train_evaluation.csv,30490,1947


## Column names and types

Sales use float32 through the explicit target-loader contract. Other fields use loader types or pandas inference. CSV itself has no intrinsic dtypes. Consecutive ranges include every integer between the endpoints; exact names are in `pd.read_csv(path, nrows=0).columns`.

In [3]:
for name, info in audit["files"].items():
    display(Markdown(f"**{name}**"))
    display(pd.DataFrame(info["schema"]))

**calendar.csv**

,column(s),dtype
0,date,str
1,wm_yr_wk,int64
2,weekday,str
3,wday,int64
4,month,int64
5,year,int64
6,d,str
7,event_name_1,str
8,event_type_1,str
9,event_name_2,str


**sell_prices.csv**

,column(s),dtype
0,store_id,string
1,item_id,string
2,wm_yr_wk,int32
3,sell_price,float32


**sales_train_validation.csv**

,column(s),dtype
0,id,string
1,item_id,string
2,dept_id,string
3,cat_id,string
4,store_id,string
5,state_id,string
6,d_1 … d_1913 (all consecutive),float32 (explicit target loader)


**sales_train_evaluation.csv**

,column(s),dtype
0,id,string
1,item_id,string
2,dept_id,string
3,cat_id,string
4,store_id,string
5,state_id,string
6,d_1 … d_1941 (all consecutive),float32 (explicit target loader)


## First five rows and missing values

Wide sales previews show static metadata plus d_1, d_2 and d_1801. Later targets are deliberately omitted. Price/calendar previews are structural records. Missing sales-target counts apply only through d_1801; the holdout is not summarized. Pandas abbreviates wide previews with `…`.

In [4]:
for name, info in audit["files"].items():
    display(Markdown(f"**{name}**"))
    display(pd.DataFrame(info["sample"]))
    print("Missingness:", info["missing"])

**calendar.csv**

,date,wm_yr_wk,weekday,wday,month,year,...,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,...,None,None,None,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,...,None,None,None,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,...,None,None,None,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,...,None,None,None,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,...,None,None,None,1,0,1


Missingness: {'scope': 'all non-sales fields', 'by_column': {'date': 0, 'wm_yr_wk': 0, 'weekday': 0, 'wday': 0, 'month': 0, 'year': 0, 'd': 0, 'event_name_1': 1807, 'event_type_1': 1807, 'event_name_2': 1964, 'event_type_2': 1964, 'snap_CA': 0, 'snap_TX': 0, 'snap_WI': 0}}


**sell_prices.csv**

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


...

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_1801
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,1.0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,0.0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,1.0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,2.0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,0.0


Missingness: {'scope': 'metadata and target d_1–d_1801 only', 'static_missing_cells': 0, 'target_missing_cells': 0}


**sales_train_evaluation.csv**

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_1801
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,1.0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,0.0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,1.0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,2.0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0.0,0.0,0.0


Missingness: {'scope': 'metadata and target d_1–d_1801 only', 'static_missing_cells': 0, 'target_missing_cells': 0}


## Identifier counts, dates and Walmart weeks

A sales row is one item/store pair, a calendar row is one date, a price row is one item/store/week. `snap_CA/TX/WI` are state flags rather than a `state_id` column.

In [5]:
identifiers = ["item_id", "store_id", "dept_id", "cat_id", "state_id", "item_store_series"]
display(pd.DataFrame([
    {"dataset": name, **{c: info["unique"].get(c, "—") for c in identifiers}}
    for name, info in audit["files"].items()
]).set_index("dataset"))
for name, info in audit["files"].items():
    print(name, info["coverage"])

,item_id,store_id,dept_id,cat_id,state_id,item_store_series
dataset,,,,,,
calendar.csv,—,—,—,—,—,—
sell_prices.csv,3049,10,—,—,—,—
sales_train_validation.csv,3049,10,7,3,3,30490
sales_train_evaluation.csv,3049,10,7,3,3,30490


calendar.csv {'first_week_key': 11101, 'last_week_key': 11621, 'distinct_week_keys': 282, 'first_date': '2011-01-29', 'last_date': '2016-06-19', 'first_day': 'd_1', 'last_day': 'd_1969'}
sell_prices.csv {'first_week_key': 11101, 'last_week_key': 11621, 'distinct_week_keys': 282}
sales_train_validation.csv {'first_day': 'd_1', 'last_day': 'd_1913', 'first_date': '2011-01-29', 'last_date': '2016-04-24'}
sales_train_evaluation.csv {'first_day': 'd_1', 'last_day': 'd_1941', 'first_date': '2011-01-29', 'last_date': '2016-05-22'}


## Key constraints, history overlap and joins

- Sales `(item_id, store_id)` is unique independently in both training files.
- Calendar `d` and `date` are unique. Prices `(item_id, store_id, wm_yr_wk)` is unique.
- Sales day headers must become a row-level `d` before the many-to-one calendar join. Calendar supplies `wm_yr_wk` for the many-to-one weekly-price join.
- Evaluation is the sole canonical source. Its extra 28 days are the reserved final holdout. The validation file exists here to verify the common history; it is never appended.
- Literal training/submission `id` suffixes differ. Canonical `series_id = item_id + '__' + store_id` identifies the underlying series.

Known calendar dates extend to d_1969. Submission validation dates map to d_1914–d_1941 and evaluation dates to d_1942–d_1969; competition submission windows do not override our project holdout policy.

In [6]:
display(pd.Series(audit["keys"], name="verified keys"))
print("Overlap verification:", audit["overlap_check"])

sales       item_id + store_id unique in both files
calendar            d and date independently unique
prices         item_id + store_id + wm_yr_wk unique
Name: verified keys, dtype: str

Overlap verification: {'matched': True, 'series': 30490, 'days': 1913, 'cells_compared': 58327370, 'excluded': 'id suffix differs; no holdout targets compared'}


## Select 240 diverse series

Eligibility requires the configured nonmissing-history fraction through d_1801 (default 100%); there is **no minimum positive-volume requirement**. Rank candidates using the mean of d_1437–d_1801 (365 days), break ties by series ID, and split ranks into low/medium/high terciles. With seed 42, sample 80 per tercile by round-robin across store/category strata, randomizing candidates and stratum order.

This retains sparse/all-zero candidates. Diversity is sought where feasible; tied means can straddle rank bands. Sampling is intentionally stratified rather than population-representative. Statistics and selected IDs are saved for reproducibility.

In [7]:
display(selected.groupby("volume_group").agg(
    series=("series_id", "size"), stores=("store_id", "nunique"),
    categories=("cat_id", "nunique"),
    trailing_mean_min=("mean_trailing_365", "min"),
    trailing_mean_max=("mean_trailing_365", "max")
).reindex(["low", "medium", "high"]).round(4))
print("Eligible:", selection["eligible"], "Excluded:", selection["excluded"])
print("Compromises:", selection["compromises"])
display(selected.head(5))
print("Saved: data/interim/selected_skus.csv, selection_metadata.json")
print("Dictionary: reports/dataset_dictionary.md; machine-readable audit: reports/source_audit.json")

,series,stores,categories,trailing_mean_min,trailing_mean_max
volume_group,,,,,
low,80,10,3,0.0110,0.3205
medium,80,10,3,0.3315,0.9123
high,80,10,3,0.9205,9.3973


Eligible: 30490 Excluded: 0
Compromises: ['Equal group sizes and diversity deliberately change population proportions; no population representativeness claimed.']


,series_id,item_id,store_id,dept_id,cat_id,state_id,observed_fraction,mean_history,mean_trailing_365,zero_fraction,selection_cutoff,volume_group
0,FOODS_1_004__TX_2,FOODS_1_004,TX_2,FOODS_1,FOODS,TX,1.0,7.174903,3.723288,0.335369,1801,high
1,FOODS_1_019__WI_1,FOODS_1_019,WI_1,FOODS_1,FOODS,WI,1.0,3.516935,4.224658,0.275958,1801,high
2,FOODS_1_207__CA_1,FOODS_1_207,CA_1,FOODS_1,FOODS,CA,1.0,1.295391,1.421918,0.378679,1801,high
3,FOODS_2_035__TX_3,FOODS_2_035,TX_3,FOODS_2,FOODS,TX,1.0,0.634092,1.909589,0.704053,1801,high
4,FOODS_2_051__WI_1,FOODS_2_051,WI_1,FOODS_2,FOODS,WI,1.0,0.952249,1.558904,0.639089,1801,high


Saved: data/interim/selected_skus.csv, selection_metadata.json
Dictionary: reports/dataset_dictionary.md; machine-readable audit: reports/source_audit.json
